In [ ]:
from init_spark import spark_session
from pyspark.sql.functions import *
from pyspark.sql import functions as F
from functools import reduce
from pyspark.sql.window import Window

In [ ]:
spark = spark_session()

# Create the tables for case 1

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

# -------------------------
# USERS 10M Rows without Skew
# -------------------------
n_users = 10_000_000

users = pd.DataFrame({
    "user_id": np.arange(n_users),
    "segment": np.random.choice(
        ["STANDARD", "PREMIUM", "VIP"],
        size=n_users,
        p=[0.85, 0.13, 0.02]
    ),
    "country": np.random.choice(
        ["USA", "PRT", "ESP", "FRA", "BRA"],
        size=n_users,
        p=[0.40, 0.15, 0.15, 0.15, 0.15]
    )
})

# -------------------------
# STORES 50K Rows with high skew
# -------------------------
n_stores = 50_000

stores = pd.DataFrame({
    "store_id": np.arange(n_stores),
    "brand": np.random.choice(
        ["NIKE", "ADIDAS", "PUMA", "BZR", "NEWBALANCE"],
        size=n_stores,
        p=[0.30, 0.25, 0.15, 0.20, 0.10]
    ),
    "region": np.random.choice(
        ["NORTH", "SOUTH", "EAST", "WEST"],
        size=n_stores
    )
})

# -------------------------
# STORE SKEW (forte e realista)
# Top 5 lojas dominam 98.4% das transações
# -------------------------

'''
We put weights on stores, each store starts with a very low base (0.001), but 5 of them we define the weights above, making them dominant.
So, we have 50k right ? 50k - 5 = 49,995 stores with 0.001 weight each, total 49.995, and for this all stores the weight for each will be 50
if we sum all weights, 800+700+600+500+400+50 = 3050 total of weights, 
so the top 5 stores will have 800/3050 = 0.262, 700/3050 = 0.229, 600/3050 = 0.197, 500/3050 = 0.164, 400/3050 = 0.131 of the transactions respectively, and the rest of the stores will have 50/3050 = 0.016 each.

'''
n_fact = 90_000_000

store_weights = np.ones(n_stores) * 0.001  

# Top 5 stores will be dominant
store_weights[0] = 800   
store_weights[1] = 700   
store_weights[2] = 600  
store_weights[3] = 500   
store_weights[4] = 400  

# Normalize
store_weights /= store_weights.sum()

print("Verify the distribution:")
print(f"  Top 5 Stores represent: {store_weights[:5].sum()*100:.1f}% of transactions")
print(f"  Remaining {n_stores-5} stores represent: {store_weights[5:].sum()*100:.1f}%")

# -------------------------
# FACT TABLE 90M Rows
# -------------------------
fact = pd.DataFrame({
    "transaction_id": np.arange(n_fact),
    "user_id": np.random.randint(0, n_users, size=n_fact),  # uniforme, sem skew
    "store_id": np.random.choice(
        np.arange(n_stores),
        size=n_fact,
        p=store_weights
    ),
    "amount": np.random.lognormal(mean=3, sigma=1, size=n_fact)
})

print("\nReal distribution of the top 10 stores in the fact table:")
print(fact["store_id"].value_counts().head(10))
print(f"\nTotal rows in fact table: {len(fact):,}")

In [ ]:
raw_path = r"Write the raw data to parquet files"

In [ ]:
fact.to_parquet(f"{raw_path}/fact.parquet", index=False)
users.to_parquet(f"{raw_path}/users.parquet", index=False)
stores.to_parquet(f"{raw_path}/stores.parquet", index=False)

In [ ]:
spark.sparkContext.setJobDescription('Read_Fact')

fact = spark.read.parquet(r'read the data from files/fact.parquet')

In [ ]:
spark.sparkContext.setJobDescription('write_Fact')


try:
    fact.write.format("delta") \
    .mode("overwrite") \
    .save(r"write the data to delta files/fact_delta")
except Exception as e:
    print("error to generate a Delta table:", e)

In [ ]:
spark.sparkContext.setJobDescription('Read_Dim')

dim_stores = spark.read.parquet(r'read the data from files/stores.parquet')

In [ ]:
spark.sparkContext.setJobDescription('write_Dim')

try:
    dim_stores.write.format("delta") \
    .mode("overwrite") \
    .save(r"write the data to delta files/dim_stores")
except Exception as e:
    print("error to generate a Delta table:", e)

# Create tables for Case 2

In [ ]:


# =====================================================
# CASE STUDY 2 - Large-to-Large Join with Data Skew
# Fact_Orders + Fact_OrderLines
# Cluster: 2 executors x 2 cores x 1GB RAM
# =====================================================

spark.conf.set("spark.sql.shuffle.partitions", "200")

num_orders = 6_000_000

num_normal_order_lines = 3_000_000

skewed_orders = [
    (1, 10_000_000),
    (2, 8_000_000),
    (3, 7_000_000),
    (4, 6_000_000),
    (5, 4_000_000),
]



# -----------------------------
# Fact_Orders
# 1 row per order
# -----------------------------

fact_orders = (
    spark.range(1, num_orders + 1)
    .withColumnRenamed("id", "order_id")
    .withColumn("customer_id", (rand(seed=10) * 500_000).cast("int"))
    .withColumn("store_id", (rand(seed=20) * 5_000).cast("int"))
    .withColumn(
        "order_date",
        date_add(lit("2024-01-01"), (rand(seed=30) * 365).cast("int"))
    )
    .withColumn(
        "order_status",
        expr("""
            CASE
                WHEN rand(40) < 0.70 THEN 'COMPLETED'
                WHEN rand(41) < 0.85 THEN 'CANCELLED'
                ELSE 'RETURNED'
            END
        """)
    )
    .withColumn(
        "payment_method",
        expr("""
            CASE
                WHEN rand(50) < 0.50 THEN 'CARD'
                WHEN rand(51) < 0.75 THEN 'PAYPAL'
                ELSE 'BANK_TRANSFER'
            END
        """)
    )
    .withColumn(
        "channel",
        expr("""
            CASE
                WHEN rand(60) < 0.70 THEN 'ONLINE'
                ELSE 'STORE'
            END
        """)
    )
    .withColumn("order_total_amount", round(rand(seed=70) * 500 + 10, 2))
)

# -----------------------------
# Fact_OrderLines - normal lines
# -----------------------------

normal_order_lines = (
    spark.range(1, num_normal_order_lines + 1)
    .withColumnRenamed("id", "order_line_id")
    .withColumn("order_id", (rand(seed=100) * num_orders + 1).cast("long"))
    .withColumn("product_id", (rand(seed=110) * 200_000).cast("int"))
    .withColumn("quantity", (rand(seed=120) * 5 + 1).cast("int"))
    .withColumn("unit_price", round(rand(seed=130) * 200 + 1, 2))
    .withColumn("discount_amount", round(rand(seed=140) * 10, 2))
    .withColumn(
        "line_amount",
        round(col("quantity") * col("unit_price") - col("discount_amount"), 2)
    )
)

# -----------------------------
# Fact_OrderLines - skewed lines
# -----------------------------

skewed_dfs = []
current_id = num_normal_order_lines + 1

for order_id, rows in skewed_orders:
    skewed_df = (
        spark.range(current_id, current_id + rows)
        .withColumnRenamed("id", "order_line_id")
        .withColumn("order_id", lit(order_id).cast("long"))
        .withColumn("product_id", (rand(seed=210 + order_id) * 200_000).cast("int"))
        .withColumn("quantity", (rand(seed=220 + order_id) * 5 + 1).cast("int"))
        .withColumn("unit_price", round(rand(seed=230 + order_id) * 200 + 1, 2))
        .withColumn("discount_amount", round(rand(seed=240 + order_id) * 10, 2))
        .withColumn(
            "line_amount",
            round(col("quantity") * col("unit_price") - col("discount_amount"), 2)
        )
    )

    skewed_dfs.append(skewed_df)
    current_id += rows

fact_order_lines = reduce(lambda df1, df2: df1.unionByName(df2), [normal_order_lines] + skewed_dfs)


In [ ]:
# -----------------------------
# Write Delta tables
# -----------------------------
spark.sparkContext.setJobDescription('write_fact_orders')

try:
fact_orders.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(r"write the data to delta files/fact_orders")
except Exception as e:
    print("error to generate a Delta table:", e)


In [ ]:
spark.sparkContext.setJobDescription('write_fact_order_lines')
try:
    fact_order_lines.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(r"write the data to delta files/fact_order_lines")
except Exception as e:
    print("error to generate a Delta table:", e)